# City population Markov Cahain
This is a simple implementation of a Markov Chain to model city populations over time. The model assumes that the population of a city can change based on certain probabilities.
A total of 20 cities are modeled, the population of each city can increase or decrease based on defined probabilities over iterations.
```python

In [261]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
import folium
import os

dataset_path = kagglehub.dataset_download("justinboon/municipalities-of-the-netherlands")
print(f"Dataset downloaded to: {dataset_path}")

Dataset downloaded to: /Users/jadenvanrijswijk/.cache/kagglehub/datasets/justinboon/municipalities-of-the-netherlands/versions/7


In [262]:

df = pd.read_csv(os.path.join(dataset_path, 'municipalities_v7.csv'))
df = df[['municipality', 'province', 'population', 'surface_km2', 'latitude', 'longitude']]

RANDOM_SEED = 42
CITY_SAMPLE_SIZE = 20
MAX_CITY_CONNECTIONS = 6
GEN_ITERATIONS = 8
TRANSITION_WEIGHTS = {
    'distance': 0.9,  
    'population': 0.05,
    'surface_km2': 0.05,
}

print(df.size)
print(df.info())

41040
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6840 entries, 0 to 6839
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   municipality  6840 non-null   object 
 1   province      6840 non-null   object 
 2   population    376 non-null    float64
 3   surface_km2   6804 non-null   float64
 4   latitude      6840 non-null   float64
 5   longitude     6840 non-null   float64
dtypes: float64(4), object(2)
memory usage: 320.8+ KB
None


In [263]:
df = df.dropna()
df = df[df['province'] == 'Noord-Holland']
df = df[df['population'] > 0]
df = df.rename(columns={'municipality': 'city_name'})
 
df

,city_name,province,population,surface_km2,latitude,longitude
2,Aalsmeer,Noord-Holland,30792.0,32.3,52.262164,4.761922
7,Alkmaar,Noord-Holland,94906.0,31.2,52.632842,4.755037
14,Amstelveen,Noord-Holland,85135.0,44.1,52.311421,4.870087
15,Amsterdam,Noord-Holland,853312.0,219.3,52.370216,4.895168
27,Beemster,Noord-Holland,8919.0,72.1,52.547559,4.913332
32,Bergen (NH.),Noord-Holland,30075.0,119.5,52.674937,4.706395
38,Beverwijk,Noord-Holland,40052.0,20.1,52.486984,4.657447
41,Blaricum,Noord-Holland,9112.0,15.6,52.272669,5.248080
42,Bloemendaal,Noord-Holland,22077.0,45.2,52.404947,4.620185
59,Castricum,Noord-Holland,34244.0,60.4,52.545259,4.672735


In [264]:
# Helper functions

def latitude_longtitude_to_distance_km(lat1, lon1, lat2, lon2):
    """ Haversine formula to calculate distance between two lat/lon points in km.
        This function was made with AI assistance.
    """
    R = 6371  # Radius of the Earth in km
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    distance = R * c
    return distance

def visualize_chain(markov_chain):
    """Visualize cities and their connections on an interactive Folium map."""
    cities_df = markov_chain.cities_df
    adjacency_matrix = markov_chain.adjacency_matrix

    # Center map on average location of all cities
    center_lat = cities_df['latitude'].mean()
    center_lon = cities_df['longitude'].mean()
    m = folium.Map(location=[center_lat, center_lon], zoom_start=8)

    # --- Draw cities ---
    for _, city in cities_df.iterrows():
        folium.CircleMarker(
            location=[city.latitude, city.longitude],
            radius=max(3, city.population / cities_df['population'].max() * 10),  # dynamic scaling
            popup=(
                f"{city.city_name}<br>"
                f"Population: {int(city.population)}<br>"
                f"Surface: {city.surface_km2} km²"
            ),
            color='crimson' if city.city_name == markov_chain.cities_df.iloc[0]['city_name'] else 'blue',
            fill=True,
            fill_opacity=0.7,
        ).add_to(m)

    # --- Draw connections ---
    drawn_edges = set()
    for city1 in adjacency_matrix.index:
        connected = adjacency_matrix.loc[city1]
        for city2, connected_flag in connected.items():
            if connected_flag != 1 or (city2, city1) in drawn_edges:
                continue

            c1 = cities_df[cities_df['city_name'] == city1].iloc[0]
            c2 = cities_df[cities_df['city_name'] == city2].iloc[0]

            folium.PolyLine(
                locations=[
                    [c1.latitude, c1.longitude],
                    [c2.latitude, c2.longitude],
                ],
                color='black',
                weight=2,
                opacity=0.6,
            ).add_to(m)

            drawn_edges.add((city1, city2))

    return m

In [265]:
city_names = df['city_name']
distance_matrix = pd.DataFrame(index=city_names, columns=city_names)

for i, city1 in df.iterrows():
    for j, city2 in df.iterrows():
        distance = latitude_longtitude_to_distance_km(city1['latitude'], city1['longitude'], city2['latitude'], city2['longitude'])
        distance_matrix.at[city1['city_name'], city2['city_name']] = distance

In [271]:
class MarkovChain:
    def __init__(self, cities_df, n_nodes, distance_matrix, transition_weights, starting_city=None):
        self.cities_df = (
            cities_df.sample(n=n_nodes, random_state=RANDOM_SEED)
            .reset_index(drop=True)
            .copy()
        )
        self.distance_matrix = distance_matrix
        self.transition_weights = transition_weights

        if starting_city and starting_city['city_name'] not in self.cities_df['city_name'].values:
            random_index = np.random.randint(0, n_nodes)
            self.cities_df.iloc[random_index] = starting_city

        self.n_nodes = n_nodes
        self.adjacency_matrix = pd.DataFrame(
            0,
            index=self.cities_df['city_name'],
            columns=self.cities_df['city_name'],
            dtype=int,
        )

        if starting_city is not None:
            total_population = self.cities_df['population'].sum()
            self.cities_df['population'] = 0
            self.cities_df.loc[
                self.cities_df['city_name'] == starting_city['city_name'], 'population'
            ] = total_population

    def evaluate_attractiveness(self, city1, city2):
        """Compute attractiveness of city2 for someone in city1."""

        c1, c2 = city1['city_name'], city2['city_name']

        # distance normalized to [0, 1] (closer = higher factor)
        d = self.distance_matrix.at[c1, c2]
        max_d = self.distance_matrix.max().max()
        distance_factor = 1 - (d / max_d)

        # population and surface normalized to [0, 1]
        pop_factor = city2['population'] / self.cities_df['population'].max()
        surf_factor = city2['surface_km2'] / self.cities_df['surface_km2'].max()

        # weighted sum
        attractiveness = (
            self.transition_weights['distance'] * distance_factor
            + self.transition_weights['population'] * pop_factor
            + self.transition_weights['surface_km2'] * surf_factor
        )

        # normalize to [0, 1] using min–max scaling across all possible values
        min_attr = min(self.transition_weights.values())
        max_attr = max(self.transition_weights.values())
        attractiveness = (attractiveness - min_attr) / (max_attr - min_attr)
        attractiveness = np.clip(attractiveness, 0, 1)

        return attractiveness
                    
                        
    def evolve_connections(
        self,
        base_threshold=0.5,
        allow_loops=True,
        random_state=None,
        strategic_ratio=0.1,
        distance_bias=0.6,
        randomness_strength=0.3,
    ):
        """Establish realistic, random bidirectional connections.
        - Nearby cities connect more often
        - Larger cities attract more links
        - Some randomness ensures diversity
        - Caps on maximum connections per city
        """

        rng = np.random.default_rng(random_state)
        self.adjacency_matrix.loc[:, :] = 0  # reset

        city_names = self.cities_df['city_name'].values
        n = len(city_names)
        max_connections = int(n / 2)
        n_strategic = max(1, int(strategic_ratio * n))
        strategic_cities = rng.choice(city_names, n_strategic, replace=False)

        # normalize features for weighting
        norm_pop = self.cities_df['population'] / self.cities_df['population'].max()
        norm_surface = self.cities_df['surface_km2'] / self.cities_df['surface_km2'].max()

        for i, city1 in enumerate(city_names):
            for j, city2 in enumerate(city_names):
                if j <= i:
                    continue  # skip lower triangle

                if allow_loops and i == j:
                    importance = 0.5 * norm_pop.iloc[i] + 0.5 * norm_surface.iloc[i]
                    if importance > 0.6:
                        self.adjacency_matrix.at[city1, city2] = 1
                    continue

                # skip if already at max degree
                if (
                    self.adjacency_matrix.loc[city1].sum() >= max_connections
                    or self.adjacency_matrix.loc[city2].sum() >= max_connections
                ):
                    continue

                city_row_1 = self.cities_df.iloc[i]
                city_row_2 = self.cities_df.iloc[j]

                # Base attractiveness (weighted mix of distance, pop, surface)
                base_attr = self.evaluate_attractiveness(city_row_1, city_row_2)

                # Distance-based effect
                distance = self.distance_matrix.at[city1, city2]
                max_d = self.distance_matrix.max().max()
                distance_factor = 1 - (distance / max_d)  # closer → higher

                # City “hub” factor: more connections if high pop or strategic
                hub_factor = 1.2 if (city1 in strategic_cities or city2 in strategic_cities) else 1.0
                hub_factor += 0.3 * (norm_pop.iloc[i] + norm_pop.iloc[j]) / 2

                # Random noise: small random shift to break determinism
                noise = rng.normal(0, randomness_strength * 0.1)

                # Final connection probability
                prob = (
                    (distance_bias * distance_factor)
                    + (1 - distance_bias) * base_attr
                ) * hub_factor + noise

                prob = np.clip(prob, 0, 1)

                # Form connection based on threshold
                if prob >= base_threshold and rng.random() < prob:
                    self.adjacency_matrix.at[city1, city2] = 1
                    self.adjacency_matrix.at[city2, city1] = 1

        # ensure connectivity (no isolated cities)
        for city in city_names:
            if self.adjacency_matrix.loc[city].sum() == 0:
                nearest_city = self.distance_matrix.loc[city].drop(city).idxmin()
                self.adjacency_matrix.at[city, nearest_city] = 1
                self.adjacency_matrix.at[nearest_city, city] = 1


    def evolve_population(self):
        """Evolve population using attractiveness-weighted transitions.
        Larger, more attractive cities draw in more people.
        """

        # copy base adjacency
        A = self.adjacency_matrix.copy().astype(float)

        # weight connections by destination attractiveness
        for i, city_from in enumerate(self.cities_df['city_name']):
            for j, city_to in enumerate(self.cities_df['city_name']):
                if A.at[city_from, city_to] == 1:
                    attractiveness = self.evaluate_attractiveness(
                        self.cities_df.iloc[i],
                        self.cities_df.iloc[j],
                    )
                    A.at[city_from, city_to] = attractiveness
                else:
                    A.at[city_from, city_to] = 0

        # normalize columns (sum = 1 per source city)
        A = A.div(A.sum(axis=0).replace(0, 1), axis=1)

        # represent current population as vector
        v = self.cities_df['population'].values.astype(float)

        # update: new_v = A @ v (probability-weighted population movement)
        new_v = A.values @ v

        # total population conserved
        new_v *= self.cities_df['population'].sum() / new_v.sum()

        self.cities_df['population'] = new_v


# Test with starting city Amstelveen

In [267]:
amstelveen = df[df['city_name'] == 'Amstelveen'].iloc[0]
chain = MarkovChain(df, CITY_SAMPLE_SIZE, distance_matrix, TRANSITION_WEIGHTS, starting_city=amstelveen)

for i in range(GEN_ITERATIONS):
    chain.evolve_connections(base_threshold=0.7, random_state=42)
    chain.evolve_population()
    
print(chain.cities_df['population'])
    
m = visualize_chain(chain)
m

0      99098.843836
1      96659.662010
2      93325.172905
3      99764.561871
4      92125.190481
5     100391.563182
6      95962.368630
7     100408.160316
8     103778.306539
9     100781.249946
10     96876.780309
11     69525.950821
12     41935.782488
13     42253.375460
14     38599.354293
15     41065.184546
16     41682.851494
17     42605.865590
18     31224.967012
19     35149.808271
Name: population, dtype: float64


# Test without starting city and realistic population distribution

In [273]:
chain = MarkovChain(df, CITY_SAMPLE_SIZE, distance_matrix, TRANSITION_WEIGHTS)

for i in range(GEN_ITERATIONS):
    chain.evolve_connections(base_threshold=0.7, random_state=42)
    chain.evolve_population()
    
print(chain.cities_df['population'])
    
m = visualize_chain(chain)
m

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 20 is different from 21)

# Test with population distributed evenly

In [ ]:
chain = MarkovChain(df, CITY_SAMPLE_SIZE)

# Distribute population evenly over all cities
total_population = chain.cities_df['population'].sum()
even_population = total_population // CITY_SAMPLE_SIZE
chain.cities_df['population'] = even_population

for i in range(GEN_ITERATIONS):
    chain.evolve_connections()
    
m = chain.visualize_chain()
m

TypeError: MarkovChain.__init__() missing 2 required positional arguments: 'distance_matrix' and 'transition_weights'